# HydroSense-Kenya - Level 4: Data Cleaning, Analysis & Visualization
**Course:** ICS 2207 Scientific Computing | **Level:** 4 of 6 (15 marks)

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

os.makedirs('../outputs', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

weather = pd.read_csv('../data/raw/weather_daily.csv', na_values=['NA', ''])
soil    = pd.read_csv('../data/raw/soil_sensor_data.csv', na_values=['NA', ''])
params  = pd.read_csv('../data/raw/crop_zone_parameters.csv')

weather['date']      = pd.to_datetime(weather['date'])
soil['timestamp']    = pd.to_datetime(soil['timestamp'])
soil['date']         = soil['timestamp'].dt.date

print('Raw data loaded.')
print(f'Weather: {weather.shape}  Soil: {soil.shape}  Params: {params.shape}')

## 1. Data Audit - Find Every Problem

In [ ]:
print('=== MISSING VALUES ===')
print('Weather:')
print(weather.isnull().sum())
print('\nSoil:')
print(soil.isnull().sum())

In [ ]:
print('=== WEATHER ANOMALIES ===')
print(weather.describe().round(2))
print()
print('Temperature > 40C (impossible for Kenya in March):')
print(weather[weather['temperature_c'] > 40][['date','temperature_c']])
print()
print('Rainfall > 60mm (extreme):')
print(weather[weather['rainfall_mm'] > 60][['date','rainfall_mm']])

In [ ]:
print('=== SOIL SENSOR ANOMALIES ===')
print('\n1. sensor_status=CHECK rows:')
print(soil[soil['sensor_status'] == 'CHECK'][['timestamp','zone_id','soil_moisture_pct','pump_flow_lpm','sensor_status']])

print('\n2. Soil moisture < 10% (below any zone minimum):')
print(soil[soil['soil_moisture_pct'] < 10][['timestamp','zone_id','soil_moisture_pct']])

print('\n3. Tank level > 9000 L (tank capacity ~5000 L):')
print(soil[soil['tank_level_liters'] > 9000][['timestamp','zone_id','tank_level_liters']])

print('\n4. Missing soil_moisture_pct:')
print(soil[soil['soil_moisture_pct'].isnull()][['timestamp','zone_id','soil_moisture_pct']])

## 2. Data Cleaning - Every Decision Justified

In [ ]:
# ── WEATHER CLEANING ──
weather_clean = weather.copy()

# Decision 1: Missing rainfall on 2026-03-08 → impute with 0.0
# Justification: Surrounding days (Mar07=1.3, Mar09=0) indicate a dry period.
# Zero is the conservative choice - won't overestimate water supply.
weather_clean['rainfall_mm'] = weather_clean['rainfall_mm'].fillna(0.0)
print('Decision 1: Missing rainfall imputed with 0.0 mm.')

# Decision 2: Temperature 45.8°C on 2026-03-14 → replace with 7-day rolling mean
# Justification: Kenya March temps never exceed ~35°C. Clear sensor fault.
rolling_mean = weather_clean['temperature_c'].rolling(7, center=True, min_periods=3).mean()
outlier_idx  = weather_clean[weather_clean['temperature_c'] > 40].index
corrected_temp = rolling_mean[outlier_idx].values[0]
weather_clean.loc[outlier_idx, 'temperature_c'] = corrected_temp
print(f'Decision 2: Temperature outlier (45.8°C) replaced with rolling mean ({corrected_temp:.2f}°C).')

# Decision 3: Missing humidity on 2026-03-21 → impute with column mean
# Justification: Humidity changes slowly; mean is a reasonable estimate.
mean_hum = weather_clean['humidity_pct'].mean()
weather_clean['humidity_pct'] = weather_clean['humidity_pct'].fillna(mean_hum)
print(f'Decision 3: Missing humidity imputed with column mean ({mean_hum:.2f}%).')

# Decision 4: Retain 85mm rainfall on 2026-03-26, add flag
# Justification: Physically plausible storm event for Kenya's March rains.
weather_clean['rainfall_flag'] = weather_clean['rainfall_mm'].apply(
    lambda x: 'EXTREME' if x > 60 else 'OK')
print('Decision 4: 85mm rainfall retained with EXTREME flag.')

print('\nRemaining missing values in weather:')
print(weather_clean.isnull().sum())

In [ ]:
# ── SOIL CLEANING ──
soil_clean = soil.copy()

# Decision 5: Zone_B soil_moisture = 8.5% on 2026-03-25 → interpolate
# Justification: Zone B min is 24%. Reading of 8.5% is physically inconsistent
# with normal pump_flow (21.0 L/min) on the same day - sensor fault, not real drought.
fault_mask = (soil_clean['zone_id'] == 'Zone_B') & (soil_clean['soil_moisture_pct'] < 10)
soil_clean.loc[fault_mask, 'soil_moisture_pct'] = np.nan
for zone in ['Zone_A', 'Zone_B', 'Zone_C']:
    mask = soil_clean['zone_id'] == zone
    soil_clean.loc[mask, 'soil_moisture_pct'] = (
        soil_clean.loc[mask, 'soil_moisture_pct'].interpolate(method='linear'))
print('Decision 5: Zone B moisture 8.5% on day 25 replaced via linear interpolation.')

# Decision 6: sensor_status=CHECK on 2026-03-21, pump_flow=0 → set pump_flow to NaN
# Justification: Zero flow with CHECK status indicates a pump or sensor fault.
# Soil moisture reading passes range check and is retained.
check_mask = soil_clean['sensor_status'] == 'CHECK'
soil_clean.loc[check_mask, 'pump_flow_lpm'] = np.nan
print(f'Decision 6: pump_flow set to NaN for {check_mask.sum()} CHECK-status row(s).')

# Decision 7: Tank level 9900 L on 2026-03-14, Zone_C → replace with zone mean
# Justification: Tank capacity is ~5000L. 9900L is physically impossible - sensor spike.
tank_fault = (soil_clean['zone_id'] == 'Zone_C') & (soil_clean['tank_level_liters'] > 9000)
zone_c_mean_tank = soil_clean[
    (soil_clean['zone_id'] == 'Zone_C') & (soil_clean['tank_level_liters'] < 9000)
]['tank_level_liters'].mean()
soil_clean.loc[tank_fault, 'tank_level_liters'] = round(zone_c_mean_tank)
print(f'Decision 7: Tank spike (9900L) replaced with Zone C mean ({zone_c_mean_tank:.0f}L).')

# Decision 8: Interpolate any remaining missing moisture values
for zone in ['Zone_A', 'Zone_B', 'Zone_C']:
    mask = soil_clean['zone_id'] == zone
    soil_clean.loc[mask, 'soil_moisture_pct'] = (
        soil_clean.loc[mask, 'soil_moisture_pct'].interpolate(method='linear'))
print('Decision 8: Any remaining NaN moisture values interpolated.')

print('\nRemaining missing values in soil:')
print(soil_clean.isnull().sum())

In [ ]:
# Save cleaned datasets
weather_clean.to_csv('../data/processed/weather_clean.csv', index=False)
soil_clean.to_csv('../data/processed/soil_clean.csv', index=False)
print('Cleaned datasets saved to data/processed/')

## 3. Descriptive Statistics

In [ ]:
print('=== WEATHER: Descriptive Statistics (cleaned) ===')
print(weather_clean[['rainfall_mm','temperature_c','humidity_pct',
                      'wind_speed_mps','solar_index']].describe().round(3))

print('\n=== SOIL MOISTURE by Zone (cleaned) ===')
for zone in ['Zone_A', 'Zone_B', 'Zone_C']:
    zp   = params[params['zone_id'] == zone].iloc[0]
    data = soil_clean[soil_clean['zone_id'] == zone]['soil_moisture_pct']
    stress_days = (data < zp['min_moisture_pct']).sum()
    print(f'\n{zone} ({zp["crop_type"]}):')
    print(f'  Mean={data.mean():.2f}%  Median={data.median():.2f}%  Std={data.std():.2f}%')
    print(f'  Min={data.min():.2f}%   Max={data.max():.2f}%')
    print(f'  Target={zp["target_moisture_pct"]}%  Min threshold={zp["min_moisture_pct"]}%')
    print(f'  Stress days (below min): {stress_days} of 30')

## 4. Scientific Visualisations

In [ ]:
# ── PLOT 1: Rainfall and Temperature - Raw vs Cleaned ──
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

days = range(30)
axes[0].bar(days, weather['rainfall_mm'].fillna(0), color='lightblue', label='Raw', alpha=0.9)
axes[0].bar(days, weather_clean['rainfall_mm'], color='steelblue', alpha=0.6, label='Cleaned')
axes[0].axhline(60, color='red', linestyle='--', linewidth=0.8, label='Extreme threshold (60mm)')
axes[0].set_ylabel('Rainfall (mm)')
axes[0].set_title('Plot 1: Daily Rainfall - Raw vs Cleaned (30th March 2026)')
axes[0].legend()

axes[1].plot(days, weather['temperature_c'], 'r--', label='Raw temperature', alpha=0.7)
axes[1].plot(days, weather_clean['temperature_c'], 'r-', linewidth=2, label='Cleaned temperature')
axes[1].axhline(40, color='orange', linestyle=':', linewidth=1, label='Plausibility ceiling (40°C)')
axes[1].set_ylabel('Temperature (°C)')
axes[1].set_xlabel('Day of 30th March 2026')
axes[1].set_title('Temperature - Outlier on Day 14 Visible in Raw Data')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/level4_plot1_rainfall_temp.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 1 saved.')

**Plot 1 Interpretation:** The raw temperature data shows a clear sensor fault on day 14 (45.8°C - impossible for central Kenya in March). After cleaning, the corrected value (~24°C) aligns with surrounding days. The 85mm rainfall spike on day 26 is retained as a physically plausible storm event.

In [ ]:
# ── PLOT 2: Soil Moisture by Zone with Thresholds ──
T_c   = weather_clean['temperature_c'].values
W_c   = weather_clean['wind_speed_mps'].values
Sol_c = weather_clean['solar_index'].values
H_c   = weather_clean['humidity_pct'].values
et_c  = np.maximum(0.0, 0.12*T_c + 0.35*W_c + 2.4*Sol_c - 0.025*H_c)

colors = {'Zone_A': '#E74C3C', 'Zone_B': '#2ECC71', 'Zone_C': '#3498DB'}

fig, ax = plt.subplots(figsize=(13, 5))
for zone in ['Zone_A', 'Zone_B', 'Zone_C']:
    zp   = params[params['zone_id'] == zone].iloc[0]
    data = soil_clean[soil_clean['zone_id'] == zone]['soil_moisture_pct'].values
    ax.plot(range(30), data, color=colors[zone], marker='o', markersize=3,
            linewidth=1.5, label=f'{zone} ({zp["crop_type"]})')
    ax.axhline(zp['min_moisture_pct'],    color=colors[zone], linestyle=':', linewidth=0.9, alpha=0.7)
    ax.axhline(zp['target_moisture_pct'], color=colors[zone], linestyle='--', linewidth=0.9, alpha=0.7)

ax.set_xlabel('Day of 30th March 2026')
ax.set_ylabel('Soil Moisture (%)')
ax.set_title('Plot 2: Soil Moisture by Zone - Dotted=stress threshold, Dashed=target')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/level4_plot2_soil_moisture.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 2 saved.')

**Plot 2 Interpretation:** All zones show a declining moisture trend across March. Zone C (maize) approaches its stress threshold (20%) between days 18–25. The heavy rain on day 26 provides visible recovery across all zones. Zone A (tomato) remains closest to its target throughout.

In [ ]:
# ── PLOT 3: ET vs Rainfall - Net Water Balance ──
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

axes[0].plot(range(30), et_c, 'orange', linewidth=2, label='ET (mm/day)')
axes[0].fill_between(range(30), et_c, alpha=0.3, color='orange')
axes[0].axhline(et_c.mean(), color='red', linestyle='--',
                label=f'Mean ET = {et_c.mean():.2f} mm/day')
axes[0].set_ylabel('ET (mm/day)')
axes[0].set_title('Plot 3a: Daily Evapotranspiration - 30th March 2026')
axes[0].legend()

net = weather_clean['rainfall_mm'].values - et_c
axes[1].bar(range(30), net, color=['steelblue' if v >= 0 else 'coral' for v in net])
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_ylabel('Net Water Balance (mm/day)')
axes[1].set_xlabel('Day of 30th March 2026')
axes[1].set_title('Plot 3b: Net Daily Water Balance (Rainfall − ET) - Blue=surplus, Red=deficit')

print(f'Deficit days (rainfall < ET): {(net < 0).sum()}')
print(f'Total ET: {et_c.sum():.1f} mm   Total rainfall: {weather_clean["rainfall_mm"].sum():.1f} mm')

plt.tight_layout()
plt.savefig('../outputs/level4_plot3_et_balance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 3 saved.')

**Plot 3 Interpretation:** Total March ET (~130mm) significantly exceeds rainfall on 23 of 30 days. ET peaks on warm, sunny, windy days. The large surplus on day 26 (85mm storm) temporarily offsets accumulated deficits but is insufficient to fully recover the cumulative shortfall.

In [ ]:
# ── PLOT 4: Correlation Heatmap ──
combined = weather_clean[['rainfall_mm','temperature_c','humidity_pct',
                           'wind_speed_mps','solar_index']].copy()
combined['ET_mm'] = et_c

# Daily average moisture across all zones
avg_moisture = []
dates = sorted(soil_clean['date'].unique())
for d in dates:
    avg_moisture.append(soil_clean[soil_clean['date']==d]['soil_moisture_pct'].mean())
combined['avg_moisture'] = avg_moisture

corr = combined.corr()
labels = ['Rainfall','Temperature','Humidity','Wind Speed','Solar Index','ET','Avg Moisture']

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label='Pearson r')
ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=9)
ax.set_yticklabels(labels, fontsize=9)
for i in range(len(labels)):
    for j in range(len(labels)):
        col = 'white' if abs(corr.iloc[i,j]) > 0.6 else 'black'
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=8, color=col)
ax.set_title('Plot 4: Correlation Heatmap - Weather Variables, ET, Avg Soil Moisture')
plt.tight_layout()
plt.savefig('../outputs/level4_plot4_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 4 saved.')

**Plot 4 Interpretation:** ET shows strong positive correlation with temperature and solar index. Humidity negatively correlates with ET (high humidity suppresses evaporation). Avg soil moisture correlates positively with rainfall and negatively with ET - confirming the water balance model captures the dominant physical relationships.

In [ ]:
# ── PLOT 5: Tank Level and Pump Power ──
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

for zone in ['Zone_A', 'Zone_B', 'Zone_C']:
    zdata = soil_clean[soil_clean['zone_id'] == zone].reset_index(drop=True)
    axes[0].plot(range(len(zdata)), zdata['tank_level_liters'],
                 color=colors[zone], marker='s', markersize=3, label=zone)
    axes[1].plot(range(len(zdata)), zdata['pump_power_watts'],
                 color=colors[zone], marker='^', markersize=3, label=zone)

axes[0].set_ylabel('Tank Level (litres)')
axes[0].set_title('Plot 5a: Water Tank Level by Zone across 30th March 2026')
axes[0].legend()
axes[1].set_ylabel('Pump Power (Watts)')
axes[1].set_xlabel('Day of 30th March 2026')
axes[1].set_title('Plot 5b: Pump Electrical Power Consumption by Zone')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/level4_plot5_pump_energy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 5 saved.')

**Plot 5 Interpretation:** Tank levels decline steadily across March. Zone C (largest area, 180m²) drains fastest. Pump power is stable (430–510W range), indicating consistent operating pressure. Without optimised scheduling, tank reserves will be critically low by month-end - motivating the optimisation in Level 5.

## 5. Data Cleaning Summary

| Issue | Location | Decision | Justification |
|-------|----------|----------|---------------|
| Missing rainfall | Weather, day 8 | Imputed with 0.0 mm | Conservative; surrounding days dry |
| Temperature 45.8°C | Weather, day 14 | Replaced with 7-day rolling mean | Impossible for Kenya in March |
| Missing humidity | Weather, day 21 | Imputed with column mean | Humidity varies slowly |
| Extreme rainfall 85mm | Weather, day 26 | Retained with EXTREME flag | Physically plausible storm event |
| Soil moisture 8.5% Zone B | Soil, day 25 | Linear interpolation | Below zone minimum; pump data normal |
| sensor_status=CHECK, pump_flow=0 | Soil, Zone B day 21 | Set pump_flow to NaN | Sensor fault |
| Tank level 9900L | Soil, Zone C day 14 | Replaced with zone mean | Tank capacity ~5000L; impossible |

**Next step:** Level 5 uses these cleaned datasets for simulation and Monte Carlo analysis.